Groundwater | Flow Modeling Track

# Step 3: From Perceptual to Conceptual – MODFLOW Fundamentals

| **Core content** | ~10 minutes |
|:--|:--|
| **Optional: Transport & advanced grids** | +5 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Explain** how MODFLOW 6 discretizes aquifer systems using finite-difference/finite-volume methods
2. **Identify** which MODFLOW 6 packages represent different components of the Limmat Valley perceptual model
3. **Describe** the governing equation that MODFLOW solves and its key assumptions
4. **Recognize** the importance of unit consistency in groundwater modeling

## Prerequisites

Before starting this notebook, you should have:
- Completed [2_perceptual_model.ipynb](2_perceptual_model.ipynb) and developed a perceptual model of the Limmat Valley
- Basic understanding of Darcy's Law and hydraulic head
- Familiarity with water balance concepts

---

In [ ]:
# Setting up the notebook
import sys

# Add the _SUPPORT/src to the system path
sys.path.append('../../_SUPPORT/src')

## Introduction

In the previous step, we built a **perceptual model**: a descriptive, high-level understanding of the Limmat Valley aquifer system. Now, we translate that understanding into a **conceptual model**—the bridge between our mental picture and a numerical simulation.

This involves making concrete decisions:
- **Which software?** We use MODFLOW 6, the current USGS standard for groundwater modeling
- **How to discretize?** Dividing continuous space into computational cells (including flexible grids)
- **Which processes?** Selecting packages to represent rivers, wells, recharge, transport, etc.

> 🌟 **Why MODFLOW 6?**
>
> MODFLOW 6 is the latest generation of MODFLOW, released by the USGS in 2017 and now the standard for groundwater modeling:
> *   **Integrated modeling:** Flow (GWF), solute transport (GWT), and heat transport (GWE) in one framework
> *   **Flexible grids:** Supports structured grids, Voronoi grids, and fully unstructured meshes
> *   **Open-source & free:** Transparent, reproducible, and trusted by regulators worldwide
> *   **Modern architecture:** Python-friendly via FloPy, with parallel computing support
> *   **Active development:** Regular updates with new capabilities (GWE released 2024)
>
> In this course, we use MODFLOW 6 for all modeling: groundwater flow, solute transport, and heat transport.

## 1 - The Governing Equation

Before discussing discretization, it's important to understand **what equation MODFLOW actually solves**. MODFLOW simulates groundwater flow by solving the **3D groundwater flow equation**, which combines Darcy's Law with the principle of mass conservation:

$$\frac{\partial}{\partial x}\left(K_x \frac{\partial h}{\partial x}\right) + \frac{\partial}{\partial y}\left(K_y \frac{\partial h}{\partial y}\right) + \frac{\partial}{\partial z}\left(K_z \frac{\partial h}{\partial z}\right) + W = S_s \frac{\partial h}{\partial t}$$

Where:
- $h$ is hydraulic head (m)
- $K_x, K_y, K_z$ are hydraulic conductivities in each direction (m/s or m/d)
- $W$ represents sources and sinks—recharge, pumping, river leakage (1/s or 1/d)
- $S_s$ is **specific storage** (1/m)—the volume of water released from a unit volume of aquifer per unit decline in head, due to compression of water and aquifer skeleton
- $t$ is time (s or d)

For **steady-state** conditions (no change in storage over time), the right side equals zero:

$$\nabla \cdot (K \nabla h) + W = 0$$

This is the equation we solve for our initial Limmat Valley model.

**Key assumptions built into MODFLOW:**
- **Darcy's Law is valid:** Flow is laminar (slow, viscous-dominated flow where fluid particles move in parallel layers—valid when Reynolds number Re < 1–10, which is satisfied in most aquifers)
- **Fluid density is constant:** No saltwater intrusion or high-concentration effects
- **REV assumption:** The aquifer can be represented at the scale of model cells. REV stands for *Representative Elementary Volume*—the minimum volume over which averaged properties (like porosity and conductivity) become meaningful. See the [REV demo notebook](../../THEORY/_demos/explore_porosity_and_REV.ipynb) for an interactive exploration.

> ⚠️ **Critical: Unit Consistency**
>
> MODFLOW is **unit-agnostic**—it doesn't enforce any particular unit system. You must ensure all inputs are consistent. Common choices:
>
> | Property | Typical Units | Our Model |
> |----------|--------------|-----------|
> | Length | meters (m) | m |
> | Time | days (d) | d |
> | Hydraulic conductivity | m/d | m/d |
> | Pumping rate | m³/d | m³/d |
> | Recharge | m/d | m/d |
> | Specific storage | 1/m | 1/m |
>
> **A common error:** Mixing m/s (often from literature) with m/d (model input) without conversion. Always check units when entering parameter values!

## 2 - Discretizing the World: The MODFLOW Grid

The governing equation cannot be solved analytically for real-world aquifers with complex boundaries and heterogeneous properties. Instead, we use the **finite-difference method**—a numerical technique that approximates continuous equations by calculating values at discrete grid points. The domain is divided into cells, and the derivatives in the governing equation are replaced by differences between neighboring cell values.

**Traditional structured grids:**
*   **Layers:** Vertical divisions (geological units or computational slices)
*   **Rows & Columns:** Horizontal divisions forming rectangular cells

Each block is called a **cell**, and the model calculates a single hydraulic head value for each cell.

**Flexible grids in MODFLOW 6:**

MODFLOW 6 also supports **unstructured grids** that can adapt to complex geometries:

| Grid Type | Description | Best For |
|-----------|-------------|----------|
| **DIS** | Traditional structured (rows/columns/layers) | Simple domains, beginners |
| **DISV** | Voronoi or quadtree grids (layered) | Local refinement near wells/rivers |
| **DISU** | Fully unstructured | Complex 3D geology |

For our Limmat Valley model, we'll use **DISV (Voronoi grids)** to refine resolution near the rivers and Hardhof well field while keeping the rest of the domain coarser—balancing accuracy and computational efficiency.

In [ ]:
from grid_demo import show_grid_comparison

show_grid_comparison()

## 3 - Optional: Transport Modeling in MODFLOW 6

> **📘 This section is optional.** Transport modeling (GWT, GWE) will be covered in detail in the transport track of the project notebooks. This overview is provided for context.

While the GWF model calculates *where* and *how much* water flows, it doesn't tell us what's *in* the water or how temperature changes. MODFLOW 6 includes dedicated models for transport:

### Solute Transport: GWT Model

The **Groundwater Transport (GWT)** model simulates the movement of dissolved substances—contaminants, tracers, or nutrients. Key processes:

*   **Advection:** Transport with bulk groundwater flow
*   **Dispersion:** Spreading due to mechanical mixing and molecular diffusion
*   **Sorption:** Chemical attachment to aquifer materials (linear, Freundlich, Langmuir isotherms)
*   **Decay/Production:** First-order or zero-order reactions

### Heat Transport: GWE Model

The **Groundwater Energy (GWE)** model (released 2024) simulates thermal energy transport—essential for geothermal applications, aquifer thermal energy storage, and understanding temperature effects on water quality.

> ⚠️ **Darcy Velocity vs. Seepage Velocity**
>
> A common source of confusion in transport modeling:
> - **Darcy velocity** (specific discharge): $q = -K \nabla h$ — a mathematical flux per unit area, not actual water speed
> - **Seepage velocity** (pore velocity): $v = q/n_e$ — the actual velocity of water moving through pores, where $n_e$ is effective porosity
>
> Transport uses **seepage velocity** because solutes move with the actual water, not the fictitious Darcy flux. For typical aquifers with $n_e \approx 0.2$, seepage velocity is ~5× faster than Darcy velocity.

**Density limitation:** Both GWT and GWE assume fluid density is constant. For saltwater intrusion or high-concentration brines where density varies significantly, specialized approaches are needed.

## 4 - The Building Blocks of MODFLOW 6: Packages

MODFLOW 6 is modular—a collection of **packages** that each handle a specific aspect of the simulation. You combine packages to represent your conceptual model.

### Essential GWF (Flow) Packages

| Package | Purpose | Limmat Valley Use |
|---------|---------|-------------------|
| **DIS/DISV** | Grid discretization | DISV for Voronoi grid |
| **NPF** | Node Property Flow (K, storage) | Aquifer hydraulic conductivity |
| **IC** | Initial Conditions | Starting head distribution |
| **STO** | Storage (Ss, Sy) | For transient simulations |
| **CHD** | Constant Head | Western outflow boundary |
| **WEL** | Wells (pumping/injection) | Hardhof wells, lateral inflow |
| **RIV** | River-aquifer interaction | Sihl and Limmat rivers |
| **RCH** | Areal recharge | Precipitation infiltration |
| **OC** | Output Control | What results to save |
| **IMS** | Iterative Model Solution | **Solver** (required!) |



> 💡 **Don't forget the solver!**
>
> Every MODFLOW model needs a **solver package** (IMS in MODFLOW 6) to iteratively solve the system of equations. The solver controls convergence criteria and iteration limits. Without it, the model won't run.

### Common Misconceptions

| Misconception | Reality |
|---------------|---------|
| "MODFLOW calculates pressure" | MODFLOW calculates **hydraulic head** ($h = z + p/\rho g$), not pressure directly |
| "One layer = one aquifer" | Layers are numerical discretization; you can use multiple layers for one hydrogeologic unit |
| "Steady state means no flow" | Steady state means $\partial h/\partial t = 0$; water still flows, but storage doesn't change |
| "Finer grid = better results" | Finer grids increase computation and can cause numerical issues if stability criteria aren't met |
| "Transmissivity replaces conductivity" | Use $K$ for 3D models; $T = K \cdot b$ is only for 2D horizontal flow |

### Translating the Limmat Valley Perceptual Model

Now that you understand the packages, here's how we translate our perceptual model from Notebook 2 into MODFLOW 6 components:

| Limmat Valley Feature | MODFLOW 6 Package | Notes |
| --------------------- | ----------------- | ----- |
| Aquifer extent & moraine boundaries | **DISV** + IDOMAIN | Inactive cells outside aquifer |
| Variable aquifer thickness | **DISV** | Layer bottom from GIS data |
| Sihl and Limmat rivers | **RIV** | Conductance from leakage coefficients (Notebook 2) |
| Areal recharge (~10% of precipitation) | **RCH** | ~0.0003 m/d = 110 mm/year |
| Hardhof pumping wells | **WEL** | Negative rates = extraction |
| Hardhof recharge basins | **WEL** | Positive rates = injection |
| Lateral inflow from hills | **WEL** or **GHB** | Estimated flux from Notebook 2 |
| Western outflow boundary | **CHD** | Fixed head at ~390 m a.s.l. |
| Aquifer hydraulic conductivity | **NPF** | K ≈ 10⁻² m/s = 864 m/d |

The flux estimates from Notebook 2 (leakage coefficients, recharge rates, lateral inflows) become direct inputs to these packages.

## Summary

In this notebook, you learned:

- **The governing equation** that MODFLOW solves (3D groundwater flow equation based on Darcy's Law and mass conservation)
- **Discretization options** including flexible Voronoi grids (DISV) for local refinement
- **How to translate** perceptual model components into MODFLOW 6 packages
- **Transport capabilities** with integrated GWT (solute) and GWE (heat) models
- **Critical importance** of unit consistency in all model inputs

You're now ready to implement these concepts in code!

---

## Documentation Resources

Bookmark these—you'll need them throughout the course:

- **MODFLOW 6:** [modflow6.readthedocs.io](https://modflow6.readthedocs.io/en/stable/)
- **FloPy:** [flopy.readthedocs.io](https://flopy.readthedocs.io/en/stable/)
- **MODFLOW 6 Examples:** [modflow6-examples.readthedocs.io](https://modflow6-examples.readthedocs.io/)

---

## Next Steps

In the next notebook, we translate our conceptual model into a working MODFLOW 6 simulation using FloPy.

**Continue to:** [4_model_implementation.ipynb](4_model_implementation.ipynb) – Building the Limmat Valley Model

**Review if needed:** [2_perceptual_model.ipynb](2_perceptual_model.ipynb) – Perceptual Model